## Analysing SegFormer segmentation model
---

In this notebook, we are going to fine-tune SegFormerForSemanticSegmentation on a custom semantic segmentation dataset. In semantic segmentation, the goal for the model is to label each pixel of an image with one of a list of predefined classes.

## Imports 
---

In [ ]:
#external
from tqdm.notebook import tqdm

#model
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import torch
from torch.utils.data import Dataset, DataLoader

#metrics
import evaluate

import torch.nn.functional as F

#utils
from src.utils.dataset import load_foodseg103, decode_image_from_bytes
from src.utils.visualization import predict_random_images

#constants
from src.constants.category_id import CATEGORY_ID

## Testing on custom dataset
---

In [2]:
IMAGE_SIZE = 512

### Defining Dataset

In [3]:
class SemanticSegmentationFoodDataset(Dataset):
    def __init__(self, image_processor:SegformerImageProcessor, sample_size:int=None, data_type:str="train"):
        image_dataset = load_foodseg103(type=data_type, sample_size=sample_size)
        self.image_dataset = image_dataset
        self.image_processor = image_processor

    def __len__(self):
        return self.image_dataset.shape[0]
    
    def __getitem__(self, index):
        image_information = self.image_dataset.loc[index]
        image_decoded = decode_image_from_bytes(image_information["image"])
        mask = decode_image_from_bytes(image_information["label"])
        encoded_inputs = self.image_processor(image_decoded, mask, return_tensors="pt")
        for k,v in encoded_inputs.items():
          encoded_inputs[k].squeeze_() # remove batch dimension
        return encoded_inputs

In [ ]:
image_processor = SegformerImageProcessor(
    do_reduce_labels=False,
    size={"height": IMAGE_SIZE, "width": IMAGE_SIZE}
)

In [ ]:
train_dataset = SemanticSegmentationFoodDataset(image_processor, data_type="train")
validation_dataset = SemanticSegmentationFoodDataset(image_processor, data_type="validation")

In [5]:
print(f"Number of training examples: {train_dataset.__len__()}")
print(f"Number of training examples: {validation_dataset.__len__()}")

Number of training examples: 4983
Number of training examples: 2135


### Testing random example

In [6]:
image_example = train_dataset.__getitem__(0)
print(f"Image shape: {image_example["pixel_values"].shape}")
print(f"Image labels: {[CATEGORY_ID.get(i.item(), 'Unknown') for i in image_example["labels"].squeeze().unique()]}")

Image shape: torch.Size([3, 512, 512])
Image labels: ['background', 'chicken duck', 'rice', 'snow peas']


### Model definition

In [7]:
num_classes = len(CATEGORY_ID)

In [8]:
food_model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0", #smaller model
    num_labels=num_classes,
    id2label=CATEGORY_ID,
    label2id={v: k for k, v in CATEGORY_ID.items()},
)

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b0 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


For this custom dataset training we are going to freeze all layers, except the model head. This is done to make training faster and efficient for a small image set.

In [9]:
for param in food_model.base_model.parameters():
    param.requires_grad = False

### Model training

In [10]:
lr = 0.0001
batch_size = 8
num_epochs = 5

In [11]:
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_dataloader = DataLoader(validation_dataset, batch_size=batch_size)

In [12]:
optimizer = torch.optim.AdamW(food_model.parameters(), lr=lr)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
food_model.to(device);

In [13]:
metric = evaluate.load("mean_iou")

In [ ]:
food_model.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for idx, batch in enumerate(tqdm(train_dataloader)):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad()
        outputs = food_model(pixel_values=pixel_values, labels=labels)
        loss, logits = outputs.loss, outputs.logits

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

        with torch.no_grad():
            upsampled_logits = F.interpolate(logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
            predicted = upsampled_logits.argmax(dim=1)
            metric.add_batch(predictions=predicted.detach().cpu().numpy(), references=labels.detach().cpu().numpy())
        
    if epoch % 10 == 0:  
        metrics = metric._compute(
                predictions=predicted.cpu(),
                references=labels.cpu(),
                num_labels=num_classes,
                ignore_index=0,
                reduce_labels=False
            )
        print("Epoch:", epoch, "Loss:", round(loss.item(), 4), "Mean_iou:", round(metrics["mean_iou"], 4), "Mean accuracy:", round(metrics["mean_accuracy"], 4))

  0%|          | 0/623 [00:00<?, ?it/s]

c:\Users\ferna\AppData\Local\Programs\Python\Python312\Lib\site-packages\datasets\features\image.py:347: UserWarning: Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
  warnings.warn(f"Downcasting array dtype {dtype} to {dest_dtype} to be compatible with 'Pillow'")


### Testing in new image

In [ ]:
food_model.eval();

In [ ]:
predict_random_images(food_model, image_processor, validation_dataset, num_images=10)

## References
[1] https://github.com/NielsRogge/Transformers-Tutorials/tree/master/SegFormer